# NALAPRO Project — Part 3
## Domain-Adaptive MLM Pre-training, then Classification Fine-tuning

---

## Tools used
| Tool | Purpose |
|------|---------|
| `transformers` | `AutoModelForMaskedLM`, `DataCollatorForLanguageModeling`, `Trainer` |
| `datasets` | Dataset pipeline for MLM chunking |
| `evaluate` | Accuracy & macro-F1 |
| `scikit-learn` | Train/val/test split, metrics |
| `PyTorch` | Compute engine |
| `Claude (Anthropic)` | Scaffolding and explanation |

**Reference:** *Hands-On Large Language Models* — https://github.com/HandsOnLLM/Hands-On-Large-Language-Models  
**Academic reference:** Gururangan et al., 'Don't Stop Pretraining', ACL 2020.

---

## 📖 The two-stage approach

```
Stage 1 — MLM (unsupervised, no labels used)
  bert-base-uncased (pre-trained on Wikipedia + BookCorpus)
        ↓
  Continue training with Masked Language Modelling on 20-Newsgroups text
        ↓
  bert-base-uncased-NEWSGROUPS  ← our domain-adapted checkpoint

Stage 2 — Classification fine-tuning (supervised)
  bert-base-uncased-NEWSGROUPS
        ↓
  Attach classification head, fine-tune exactly as in Part 2
        ↓
  Final classifier → compare to Part 2
```



This technique is called **Domain-Adaptive Pre-Training (DAPT)**.

---


## Setup


In [ ]:
%pip install -q torch transformers datasets evaluate scikit-learn accelerate wandb


---
## Experiment tracking (in-notebook statistics)

**Stage 1 — MLM pre-training**
- Training loss per step (from `mlm_trainer.state.log_history`)
- Final MLM cross-entropy loss and perplexity on the training corpus

**Stage 2 — Classification fine-tuning**
- Per-epoch validation loss, accuracy, and macro-F1 (from `cls_trainer.state.log_history`)
- Parameter counts: total, trainable, classifier head weights and biases
- Classifier head weight distribution histogram and per-class bias bar chart


In [ ]:
import os, random, math
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_ID = "bert-base-uncased"
MAX_LEN  = 256


In [ ]:
# ── Google Drive setup (Google Colab only) ────────────────────────────────────────
# Mounts Drive so CSVs and plots persist across sessions and are visible to Part 4.
# MLM and classification checkpoints stay in fast local /content/ storage.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    NALAPRO_DIR = '/content/drive/MyDrive/NALAPRO'
    os.makedirs(NALAPRO_DIR, exist_ok=True)
    os.chdir(NALAPRO_DIR)
    _IN_COLAB = True
    CKPT_BASE = '/content'   # fast local storage for model checkpoints
    print(f'Google Colab detected.')
    print(f'Working directory (CSVs/plots) → {NALAPRO_DIR}')
    print(f'Checkpoint base (fast local)   → {CKPT_BASE}')
except ModuleNotFoundError:
    _IN_COLAB = False
    CKPT_BASE = os.getcwd()
    print(f'Not in Colab. Working directory / checkpoint base: {CKPT_BASE}')

---
## 1 · Data loading

In [ ]:
from sklearn.datasets import fetch_20newsgroups

REMOVE = ("headers", "footers", "quotes")
ng_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SEED)
ng_test  = fetch_20newsgroups(subset="test",  remove=REMOVE, random_state=SEED)

class_names = ng_train.target_names
NUM_CLASSES = len(class_names)

# Collect only non-empty training documents for MLM
mlm_texts = [t for t in ng_train.data if t and len(t.strip()) > 10]
print(f"Documents for MLM pre-training : {len(mlm_texts):,}")
print(f"Total characters               : {sum(len(t) for t in mlm_texts):,}")
print(f"Average document length (chars): {sum(len(t) for t in mlm_texts)//len(mlm_texts):,}")


---
## Stage 1 · MLM pre-training

### Step 1 — Tokenise for MLM


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
mlm_ds    = Dataset.from_dict({"text": mlm_texts})

def tokenize_no_pad(batch):
    """
    Tokenise without padding or truncation.
    return_special_tokens_mask=True is needed by DataCollatorForLanguageModeling
    to know which positions are [CLS]/[SEP] (those should NOT be masked).
    """
    return tokenizer(
        batch["text"],
        truncation=False,      # don't truncate — we'll chunk below
        return_special_tokens_mask=True,
    )

tok_mlm = mlm_ds.map(tokenize_no_pad, batched=True, remove_columns=["text"])
print("Tokenised dataset:", tok_mlm)


In [ ]:
def chunk_texts(examples, block_size=MAX_LEN):
    """
    Concatenate all token IDs from the batch into a flat list,
    then split into fixed-length blocks.
    This is the standard HuggingFace recipe for MLM pre-training.
    """
    # Flatten: list of lists → one big list per key
    flat = {k: sum(examples[k], []) for k in examples}
    total = len(flat["input_ids"])
    # Discard a partial final block so every block is exactly block_size
    total = (total // block_size) * block_size
    return {
        k: [flat[k][i: i + block_size] for i in range(0, total, block_size)]
        for k in flat
    }

lm_ds = tok_mlm.map(chunk_texts, batched=True, batch_size=1000)
print(f"\nNumber of {MAX_LEN}-token chunks (= training examples): {len(lm_ds):,}")


### Step 2 — DataCollatorForLanguageModeling

The collator does the dynamic masking at each training step:

1. With probability **0.15** (BERT's original masking rate), a token is selected.
2. Of the selected tokens:
   - **80 %** are replaced with `[MASK]`
   - **10 %** are replaced with a random token (keeps BERT robust to noisy input)
   - **10 %** are kept unchanged (forces BERT to maintain the original representation)


In [ ]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)

mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,   # mask 15% of tokens per BERT's original recipe
)

total_params = sum(p.numel() for p in mlm_model.parameters())
print(f"Model: {MODEL_ID}  | Parameters: {total_params:,}")


### Step 3 — MLM Training


In [ ]:
from transformers import TrainingArguments, Trainer

mlm_args = TrainingArguments(
    output_dir           = f'{CKPT_BASE}/ckpt_part3_mlm',
    num_train_epochs     = 3,
    per_device_train_batch_size = 16,
    learning_rate        = 5e-5,
    weight_decay         = 0.01,
    warmup_steps         = 293, # Calculated from 0.10 ratio * (15621 samples / 16 batch_size) * 3 epochs
    save_strategy        = 'epoch',
    save_total_limit     = 1,
    logging_steps        = 50,
    report_to            = 'none',
    seed                  = SEED,
    fp16                 = torch.cuda.is_available(),
)

mlm_trainer = Trainer(
    model            = mlm_model,
    args             = mlm_args,
    train_dataset    = lm_ds,
    data_collator    = mlm_collator,
    processing_class = tokenizer,
)

mlm_trainer.train()

In [ ]:
# ── Perplexity as a sanity check ──────────────────────────────────────────────────
# Perplexity = exp(cross-entropy loss on masked tokens).
# Lower perplexity = better MLM predictions = more adapted to the domain.
metrics = mlm_trainer.evaluate(eval_dataset=lm_ds)
mlm_loss = metrics.get("eval_loss", float("nan"))
try:
    perplexity = math.exp(mlm_loss)
except OverflowError:
    perplexity = float("inf")
print(f"Final MLM cross-entropy loss : {mlm_loss:.4f}")
print(f"Perplexity                   : {perplexity:.2f}")
print("(Lower perplexity = better language model fit on this corpus)")


In [ ]:
MLM_CKPT = f'{CKPT_BASE}/ckpt_part3_mlm_final'
mlm_trainer.save_model(MLM_CKPT)
tokenizer.save_pretrained(MLM_CKPT)
print("Domain-adapted BERT saved to:", MLM_CKPT)

### Qualitative check — fill-mask

A great way to *show* domain adaptation is to compare what the original and adapted BERTs
predict for a masked token in a newsgroup-style sentence.


In [ ]:
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1

fill_orig    = pipeline("fill-mask", model=MODEL_ID,  tokenizer=MODEL_ID,  device=device_id)
fill_adapted = pipeline("fill-mask", model=MLM_CKPT, tokenizer=MLM_CKPT, device=device_id)

probes = [
    "I think the [MASK] operating system has the best graphics card support.",
    "NASA is planning a new [MASK] mission to explore the outer planets.",
    "The [MASK] encryption algorithm is widely used for secure communications.",
    "She plays [MASK] for the local hockey team every winter season.",
]

for probe in probes:
    print("=" * 70)
    print("Probe:", probe)
    print("\nOriginal bert-base-uncased top-5 predictions:")
    for p in fill_orig(probe)[:5]:
        print(f"   {p['token_str']:>15s}  score={p['score']:.4f}")
    print("\nAfter MLM on 20-Newsgroups top-5 predictions:")
    for p in fill_adapted(probe)[:5]:
        print(f"   {p['token_str']:>15s}  score={p['score']:.4f}")
    print()


---
## Stage 2 · Classification fine-tuning on the adapted checkpoint


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, DatasetDict
import evaluate as hf_evaluate
from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments, Trainer)

# ── Rebuild the labelled dataset (same split as Part 2) ────────────────────────────
X_tr, X_va, y_tr, y_va = train_test_split(
    ng_train.data, ng_train.target,
    test_size=0.10, stratify=ng_train.target, random_state=SEED,
)
raw_cls = DatasetDict({
    "train":      Dataset.from_dict({"text": X_tr,         "label": y_tr.tolist()}),
    "validation": Dataset.from_dict({"text": X_va,         "label": y_va.tolist()}),
    "test":       Dataset.from_dict({"text": ng_test.data,  "label": ng_test.target.tolist()}),
}).filter(lambda ex: ex["text"] is not None and len(ex["text"].strip()) > 10)

# Load the adapted tokeniser (same as original, but stored alongside the checkpoint)
tok_adapted = AutoTokenizer.from_pretrained(MLM_CKPT)

def tok_fn(batch):
    return tok_adapted(batch["text"], truncation=True, max_length=MAX_LEN)

tok_cls = raw_cls.map(tok_fn, batched=True, remove_columns=["text"])
data_collator_cls = DataCollatorWithPadding(tokenizer=tok_adapted)

# ── Load the ADAPTED checkpoint and attach a fresh classification head ─────────────
cls_model = AutoModelForSequenceClassification.from_pretrained(
    MLM_CKPT, num_labels=NUM_CLASSES
)
print("Classification model loaded from domain-adapted checkpoint.")

# ── Metrics ───────────────────────────────────────────────────────────────────────
_acc = hf_evaluate.load("accuracy")
_f1  = hf_evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": _acc.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": _f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [ ]:
cls_args = TrainingArguments(
    output_dir       = f'{CKPT_BASE}/ckpt_part3_cls',
    learning_rate    = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    num_train_epochs  = 3,
    weight_decay      = 0.01,
    eval_strategy     = 'epoch',
    save_strategy     = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    greater_is_better      = True,
    logging_steps     = 50,
    report_to         = 'none',
    seed              = SEED,
    fp16              = torch.cuda.is_available(),
)

cls_trainer = Trainer(
    model            = cls_model,
    args             = cls_args,
    train_dataset    = tok_cls['train'],
    eval_dataset     = tok_cls['validation'],
    processing_class = tok_adapted,
    data_collator    = data_collator_cls,
    compute_metrics  = compute_metrics,
)

cls_trainer.train()

In [ ]:
cls_val  = cls_trainer.evaluate(eval_dataset=tok_cls['validation'])
cls_test = cls_trainer.evaluate(eval_dataset=tok_cls['test'])
print('\n── Part 3 Stage 2 results ──────────────────────────────────')
print('Validation:', cls_val)
print('Test:      ', cls_test)

total_cls     = sum(p.numel() for p in cls_model.parameters())
trainable_cls = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
head_w_count  = cls_model.classifier.weight.numel()
head_b_count  = cls_model.classifier.bias.numel()
print(f'\nParameter breakdown:')
print(f'  Total params    : {total_cls:,}')
print(f'  Trainable       : {trainable_cls:,}  (100.0% — full fine-tune from adapted ckpt)')
print(f'  Classifier head : {head_w_count:,} weights + {head_b_count} biases')

---
## 3 · Comparison & analysis


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Parameter stats ───────────────────────────────────────────────────────────────
total_cls     = sum(p.numel() for p in cls_model.parameters())
trainable_cls = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
frozen_cls    = total_cls - trainable_cls
head_w        = cls_model.classifier.weight.numel()
head_b        = cls_model.classifier.bias.numel()

# ── Load Part 2 best run for cross-part comparison ───────────────────────────────
# part2_results.csv columns: Experiment | Test Acc | Test Macro-F1  (all string-formatted)
try:
    p2 = pd.read_csv("part2_results.csv")
    best_p2     = p2.loc[p2["Test Macro-F1"].astype(float).idxmax()]
    part2_acc   = float(best_p2["Test Acc"])
    part2_f1    = float(best_p2["Test Macro-F1"])
    part2_label = f"Part 2 — {best_p2['Experiment']}"
except Exception:
    part2_acc, part2_f1 = 0.0, 0.0
    part2_label = "Part 2 — (run Part 2 first)"

# ── Results table ─────────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {
        "Experiment"   : part2_label,
        "Total Params" : total_cls,
        "Trainable"    : total_cls,
        "Frozen"       : 0,
        "Head Weights" : head_w,
        "Head Biases"  : head_b,
        "Val F1"       : None,
        "Test Acc"     : part2_acc,
        "Test F1"      : part2_f1,
    },
    {
        "Experiment"   : "Part 3 — MLM-adapted + fine-tune",
        "Total Params" : total_cls,
        "Trainable"    : trainable_cls,
        "Frozen"       : frozen_cls,
        "Head Weights" : head_w,
        "Head Biases"  : head_b,
        "Val F1"       : cls_val["eval_macro_f1"],
        "Test Acc"     : cls_test["eval_accuracy"],
        "Test F1"      : cls_test["eval_macro_f1"],
    },
])
print("\n── Part 2 vs Part 3 comparison ─────────────────────────────────────────")
print(summary.to_string(index=False))
summary.to_csv("part3_results.csv", index=False)
print("→ Saved part3_results.csv")

# ── Bar chart: Part 2 vs Part 3 ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
experiments = ["Part 2\nBERT vanilla", "Part 3\nMLM-adapted"]
f1_scores   = [part2_f1, cls_test["eval_macro_f1"]]
bars = ax.bar(experiments, f1_scores, color=["#2196F3", "#4CAF50"], width=0.4)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Test macro-F1", fontsize=12)
ax.set_title("Effect of domain-adaptive MLM pre-training\n(same classification fine-tuning hyper-parameters)",
             fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("part3_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part3_comparison.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Training curves
# ═══════════════════════════════════════════════════════════════════════════════

def extract_log(log_history, loss_key, f1_key=None, acc_key=None):
    """Pull per-epoch eval rows from trainer.state.log_history."""
    rows = [e for e in log_history if loss_key in e]
    epochs   = [r.get("epoch") for r in rows]
    losses   = [r.get(loss_key) for r in rows]
    f1s      = [r.get(f1_key) for r in rows] if f1_key else None
    accs     = [r.get(acc_key) for r in rows] if acc_key else None
    return epochs, losses, f1s, accs

# Stage 1 — MLM: only training loss (no eval split)
mlm_train_rows = [e for e in mlm_trainer.state.log_history if "loss" in e and "eval_loss" not in e]
mlm_steps  = [r["step"] for r in mlm_train_rows]
mlm_losses = [r["loss"] for r in mlm_train_rows]

# Stage 2 — Classification
ep2, lv2, f1_2, acc2 = extract_log(
    cls_trainer.state.log_history,
    loss_key="eval_loss", f1_key="eval_macro_f1", acc_key="eval_accuracy"
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Part 3 — Training Curves", fontsize=13, y=1.01)

# Panel 1: Stage 1 MLM training loss
axes[0].plot(mlm_steps, mlm_losses, color="#FF9800", linewidth=1.5)
axes[0].set_title("Stage 1 — MLM Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

# Panel 2: Stage 2 validation loss
axes[1].plot(ep2, lv2, "o-", color="#2196F3", linewidth=2, markersize=6)
axes[1].set_title("Stage 2 — Val Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Loss")
axes[1].grid(alpha=0.3)

# Panel 3: Stage 2 val macro-F1
axes[2].plot(ep2, f1_2, "s-", color="#4CAF50", linewidth=2, markersize=6)
axes[2].set_title("Stage 2 — Val Macro-F1 per Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro-F1")
axes[2].set_ylim(0, 1.0)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("part3_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part3_training_curves.png")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Classifier head weight & bias distributions
# ═══════════════════════════════════════════════════════════════════════════════

head_weights = cls_model.classifier.weight.detach().cpu().numpy().flatten()  # (20×768,)
head_biases  = cls_model.classifier.bias.detach().cpu().numpy()              # (20,)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Part 3 — Classifier Head Parameters (after Stage 2 fine-tuning)", fontsize=12)

# Weights histogram
axes[0].hist(head_weights, bins=80, color="#4CAF50", edgecolor="white", linewidth=0.3)
axes[0].axvline(head_weights.mean(), color="red", linestyle="--", linewidth=1.2,
                label=f"mean={head_weights.mean():.4f}")
axes[0].set_title(f"Head Weights  ({len(head_weights):,} params, shape 20×768)")
axes[0].set_xlabel("Weight value")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

w_stats = (f"std={head_weights.std():.4f}  min={head_weights.min():.4f}  "
           f"max={head_weights.max():.4f}  L2={np.linalg.norm(head_weights):.3f}")
axes[0].set_xlabel(f"Weight value\n{w_stats}", fontsize=9)

# Bias bar chart (one per class)
colors = plt.cm.RdYlGn(
    (head_biases - head_biases.min()) / (head_biases.max() - head_biases.min() + 1e-9)
)
bars = axes[1].bar(range(20), head_biases, color=colors, edgecolor="white", linewidth=0.5)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Head Biases — one per class (20 params)")
axes[1].set_xlabel("Class index")
axes[1].set_ylabel("Bias value")
axes[1].set_xticks(range(20))
axes[1].set_xticklabels(range(20), fontsize=7)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("part3_head_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part3_head_distributions.png")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Per-layer parameter table for cls_model
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd

rows = []
for name, param in cls_model.named_parameters():
    data = param.detach().cpu().float().numpy().flatten()
    rows.append({
        "Layer"     : name,
        "Shape"     : str(tuple(param.shape)),
        "Params"    : param.numel(),
        "Trainable" : param.requires_grad,
        "Mean"      : f"{data.mean():.5f}",
        "Std"       : f"{data.std():.5f}",
        "L2 norm"   : f"{np.linalg.norm(data):.4f}",
    })

layer_df = pd.DataFrame(rows)
print("\n── Per-layer parameter statistics ──────────────────────────────────────")
print(layer_df.to_string(index=False))
print(f"\nTotal rows: {len(layer_df)}  |  Total params: {layer_df['Params'].sum():,}")

---
## ⚠️ Teardown — Run this before closing the notebook

> ### 🔴 STOP AND READ BEFORE YOU CLOSE
>
> **Google Colab charges per GPU-hour while the runtime is connected.**
> Even if you're not running cells, the runtime keeps billing.
>
> | Time forgotten | Approx cost (Colab Pro, T4 GPU) |
> |---|---|
> | 1 hour | ~$0.35 |
> | 1 day | ~$8 |
> | 1 week | ~$56 |
>
> **Run the two cells below, then disconnect your runtime.**

### What the teardown does
1. **Deletes model objects** from Python memory → frees GPU VRAM immediately
2. **Deletes checkpoint folders** from Colab disk → prevents disk-quota errors on next run
3. **Clears CUDA cache** → returns GPU memory to the OS
4. **Prints a checklist** → step-by-step confirmation you're fully shut down

### After running the teardown cells
Go to **Runtime → Disconnect and delete runtime** in the Colab menu bar.
That stops all billing for this session.



In [ ]:
# ── Step 1: Delete models & free GPU VRAM ────────────────────────────────────
import gc, torch

print('Freeing models from GPU VRAM…')

for _varname in ['mlm_model', 'cls_model', 'mlm_trainer', 'cls_trainer',
                 'fill_orig', 'fill_adapted']:
    if _varname in globals():
        obj = globals()[_varname]
        if hasattr(obj, 'model') and hasattr(obj.model, 'cpu'):
            obj.model.cpu()
        elif hasattr(obj, 'cpu'):
            obj.cpu()
        del globals()[_varname]
        print(f'  ✓ Deleted {_varname}')

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f'CUDA cache cleared. GPU memory still allocated: {allocated:.1f} MB')
else:
    print('No GPU detected. RAM freed via gc.collect().')

print('✓ Step 1 complete.')


In [ ]:
# ── Step 2: Delete ALL checkpoint folders from Colab disk ────────────────────
import shutil, os

ckpt_dirs = [
    f'{CKPT_BASE}/ckpt_part3_mlm',
    MLM_CKPT,                          # = {CKPT_BASE}/ckpt_part3_mlm_final
    f'{CKPT_BASE}/ckpt_part3_cls',
]

total_freed = 0
for d in ckpt_dirs:
    if os.path.isdir(d):
        size_mb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(d) for f in files
        ) / 1e6
        shutil.rmtree(d)
        total_freed += size_mb
        print(f'✓ Deleted {d}  (freed ~{size_mb:.0f} MB)')
    else:
        print(f'  {d} not found (already deleted — OK)')

print(f'\nTotal disk space freed: ~{total_freed:.0f} MB')
print()
print('=' * 65)
print('PART 3 TEARDOWN CHECKLIST')
print('=' * 65)
checks = [
    'MLM model (Stage 1) removed from GPU VRAM',
    'Classification model (Stage 2) removed from GPU VRAM',
    'fill-mask pipeline objects deleted',
    'All checkpoint directories deleted from disk',
    'No Vertex AI endpoints (Part 3 uses Colab GPU only)',
]
for c in checks:
    print(f'  ✓ {c}')
print()
print('FINAL STEP (manual):')
print('  → Colab menu: Runtime → Disconnect and delete runtime')
print('  → Verify no unexpected spend: https://console.cloud.google.com/billing')
print()
print('Part 3 teardown complete.')